# AF3 Full-Scale Pairformer & Chemical Component Dictionary

## Learning Objectives

By the end of this notebook you will be able to:

1. **Implement Pairformer at production AF3 dimensions** — 48 blocks, d_pair=128, d_single=384, device-agnostic (CPU demo + GPU-ready)
2. **Apply GPU memory optimizations** — gradient checkpointing, mixed-precision (AMP), chunked triangle attention for large N
3. **Parse CCD mmCIF entries** — extract SMILES, InChI, atom names, bond table from Chemical Component Dictionary format
4. **Build ligand atom graphs** — convert SMILES/CCD to node+edge feature tensors matching AF3's tokenization scheme
5. **Compute lDDT-PLI** — the ligand-specific scoring metric used in AF3's confidence head for small molecules
6. **Profile memory and throughput** — estimate GPU RAM requirements for different N_res and N_crop values

## Prerequisites
- Notebooks 01–03 in this directory (architecture, data pipeline, evaluation)
- PyTorch ≥ 2.0 (for `torch.compile` and AMP)
- Optional: CUDA GPU for full-scale benchmarking

## Claude Integration

> **Prompt 1 — Chunked attention**: *"Why does triangle attention have O(N²) memory? Walk through the chunked implementation step-by-step and explain what `chunk_size` controls."*

> **Prompt 2 — CCD vs SMILES**: *"What information does the CCD provide that a SMILES string alone cannot? Give three concrete examples relevant to AF3's structure prediction."*

> **Prompt 3 — Gradient checkpointing trade-off**: *"Gradient checkpointing reduces memory by recomputing activations during backward pass. Why does this slow training? When is it worth it for Pairformer?"*

> **Prompt 4 — lDDT-PLI vs lDDT**: *"How does lDDT-PLI differ from the standard lDDT metric? What specific aspect of ligand binding does it capture that lDDT misses?"*

## TL;DR — Plain English

**What this notebook does:** Scales the AF3 implementation to realistic protein sizes, adds support for ligands and nucleic acids via the Chemical Component Dictionary, and introduces memory-efficient attention.

**After this notebook you can:**
- Run the full-scale Pairformer with N=256 residues and explain the memory implications
- Use the CCD (Chemical Component Dictionary) to represent small molecules, DNA, and RNA alongside proteins
- Explain flash attention and why it is essential for training large structure models
- Discuss the engineering trade-offs in scaling AF3 to whole complexes

**Why this matters:**
- HackerRank: Scalability and memory questions (attention complexity, mixed-modality inputs) are asked at senior ML engineer level
- Isomorphic Labs: Drug discovery requires modelling protein-ligand complexes, not just proteins alone; CCD handling and flash attention are production necessities for AF3-scale models

**Time:** ~4 hours | **Difficulty:** ⭐⭐⭐⭐⭐ | **Prerequisites:** 07/01 (AF3 architecture), 07/02 (data pipeline)

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students. Beginners should treat this notebook as a conceptual first pass.

**How to study it on a first pass:**
- aim to explain the big idea of each component
- do not get stuck on every tensor shape immediately
- learn what the block is trying to accomplish biologically and geometrically

**Common traps:** drowning in implementation detail before understanding the role of Pairformer, FAPE, PAE, or diffusion in the full system.


## Canonical Resource Upgrade

Use these in order:
- [OpenFold](https://github.com/aqlaboratory/openfold) for code orientation
- [AlphaFold 2 paper](https://www.nature.com/articles/s41586-021-03819-2)
- [AlphaFold 3 paper](https://www.nature.com/articles/s41586-024-07487-w)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 07/03 (AF3 evaluation — need to measure training progress), 05/01 (transformer training fundamentals) |
| 📍 You Are Here | Module 07/04 — AF3 Full-Scale Training & CCD |
| ➡️ Next Steps | 10/01 (capstone — fine-tuning a full protein structure model) |
| 🏁 Goal | Profile and optimize Pairformer memory; set up gradient checkpointing + mixed precision + distributed training |

### 🆕 From Scratch? Start Here:
1. [PyTorch DDP tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) — distributed training basics
2. [ZeRO paper (arXiv 1910.02054)](https://arxiv.org/abs/1910.02054) — memory optimization theory
3. 05/01 (this repo) — transformer training setup
4. This notebook — full-scale training
**Time:** 10–15h | **Difficulty:** ⭐⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 07/01 (Pairformer architecture being scaled), 07/03 (metrics used during training), 05/01 (transformer training patterns)
- Used by: 10/01 (capstone uses these infrastructure patterns for fine-tuning)

## What This Notebook Teaches (Plain English)

**This is the most advanced notebook in the series.** It covers two production engineering challenges:

1. **Running Pairformer at full AF3 scale** (48 layers, large tensors) without running out of GPU memory
2. **Handling small molecules (drugs, cofactors)** using the Chemical Component Dictionary

### Required Background
- ✅ ALL previous AF3 notebooks (07/01, 07/02, 07/03)
- ✅ GPU computing concepts (CUDA, VRAM)
- ✅ PyTorch optimization (gradients, autocast)

### The Memory Problem

AF3 processes every pair of residues: for a 384-residue protein, that's 384 × 384 = 147,456 pairs. Each pair has a 128-dimensional vector. Across 48 layers with attention operations, this requires careful memory management.

| Technique | What it does | Memory savings | Speed cost |
|-----------|-------------|---------------|-----------|
| **Mixed precision (bf16)** | Use 16-bit instead of 32-bit numbers | 2× | None (faster!) |
| **Gradient checkpointing** | Recompute activations during backward pass instead of storing them | ~40× | ~20% slower training |
| **Chunked attention** | Process attention in slices instead of all at once | O(N²) vs O(N³) | Minimal |

### What is the Chemical Component Dictionary (CCD)?

SMILES strings like `c1ccc(N)cc1` describe molecular topology (atoms + bonds) but not:
- Named atom positions (N1, C2, PA, O1A) needed to match X-ray crystallography data
- Idealized 3D coordinates used to initialize structure prediction
- Leaving atom flags needed to build polymers correctly

The CCD (from the Protein Data Bank) provides all of this for 200,000+ molecules.

In [ ]:
import torch
import numpy as np

# GPU memory analysis for Pairformer
print("Pairformer Memory Analysis")
print("=" * 50)

def pairformer_memory_gb(L, c_z=128, c_s=384, n_blocks=48, dtype_bytes=2):
    """Estimate peak GPU memory for Pairformer forward pass (BF16)."""
    # Pair representation: (L, L, c_z)
    pair_mb = L * L * c_z * dtype_bytes / 1e6
    # Single representation: (L, c_s)
    single_mb = L * c_s * dtype_bytes / 1e6
    # Triangle attention intermediate: (L, L, L, n_heads) for each block
    n_heads = 4
    tri_attn_mb = L * L * L * n_heads * dtype_bytes / 1e6
    # Total per block (rough)
    per_block_mb = pair_mb * 4 + tri_attn_mb  # pair repr copies + triangle
    total_gb = (n_blocks * per_block_mb + pair_mb + single_mb) / 1e3
    return pair_mb, tri_attn_mb, total_gb

print(f"{'L':>6} {'pair (MB)':>12} {'tri_attn (MB)':>16} {'total (GB)':>12}")
for L in [64, 128, 256, 384, 512]:
    pair, tri, total = pairformer_memory_gb(L)
    print(f"{L:>6} {pair:>12.1f} {tri:>16.1f} {total:>12.2f}")
print()
print("Key: triangle attention is O(L^3) → major memory bottleneck")
print("Solution: chunk over k dimension (process L slices of size chunk_size)")

## 1. GPU Memory Analysis for Pairformer

Before writing code, understand the memory bottleneck.

**Pair representation tensor**: shape `(B, N, N, d_pair)`
- At N=256, d_pair=128: `256 × 256 × 128 × 4 bytes = 33.5 MB` per batch element
- At N=1024 (large complex): `1024 × 1024 × 128 × 4 bytes = 536 MB` — already challenging

**Triangle attention**: computes `(N, N, N)` attention weights → O(N³) compute, O(N²) memory if chunked

AF3 uses **spatial/interface cropping** (N_crop = 384 for training) to bound N.
For inference on large complexes, chunked attention is essential.

In [ ]:
import torch
import torch.nn as nn
import math

def chunked_triangle_attention(z, W_q, W_k, W_v, W_o, n_heads, chunk_size=32):
    """Memory-efficient triangle attention with chunking over k dimension.

    Standard: materializes (B, L, L, L, H) tensor  → O(L^3 * H) memory
    Chunked:  processes k in chunks of chunk_size  → O(L^2 * chunk_size * H) memory
    """
    B, L, L2, c = z.shape
    c_h = c // n_heads
    scale = math.sqrt(c_h)

    Q = W_q(z).view(B, L, L2, n_heads, c_h)  # (B, L, L, H, c_h)
    K = W_k(z).view(B, L, L2, n_heads, c_h)
    V = W_v(z).view(B, L, L2, n_heads, c_h)

    output = torch.zeros_like(Q)

    for k_start in range(0, L, chunk_size):
        k_end = min(k_start + chunk_size, L)
        K_chunk = K[:, :, k_start:k_end, :, :]  # (B, L, chunk, H, c_h)
        V_chunk = V[:, :, k_start:k_end, :, :]

        # Attention: Q (B,L,L,H,c_h) over K_chunk (B,L,chunk,H,c_h)
        # For each i: attend Q[i] over K[i, k_start:k_end]
        attn = torch.einsum('bijnhd,biknhd->bijnkh', Q.unsqueeze(3),
                            K_chunk.unsqueeze(2)) / scale
        # Accumulate (in practice: logsumexp trick for numerical stability)
        attn = torch.softmax(attn, dim=4)
        chunk_out = torch.einsum('bijnkh,biknhd->bijnhd', attn, V_chunk.unsqueeze(2))
        output += chunk_out.squeeze(3)

    return W_o(output.reshape(B, L, L, c))

torch.manual_seed(42)
B, L, c_z = 1, 16, 64
n_heads = 4
z = torch.randn(B, L, L, c_z)
W_q = nn.Linear(c_z, c_z, bias=False)
W_k = nn.Linear(c_z, c_z, bias=False)
W_v = nn.Linear(c_z, c_z, bias=False)
W_o = nn.Linear(c_z, c_z, bias=False)

out = chunked_triangle_attention(z, W_q, W_k, W_v, W_o, n_heads, chunk_size=4)
print(f"Chunked triangle attention: {z.shape} → {out.shape}")
print("Memory savings: O(L^2 * chunk) instead of O(L^3)")

## 2. Chunked Triangle Attention

Standard triangle attention materializes a `(B, N, N, N)` tensor for the attention weights.
For N=384 this is 384³ × 2 bytes = 113 MB per batch — manageable but large.
For N=1024 it becomes 2 GB — impossible on most GPUs.

**Chunking strategy**: process `chunk_size` rows of the (i,j) pair matrix at a time.
Each chunk only materializes `(B, N, chunk_size, N)` attention — reduces peak by N/chunk_size.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

class PairformerBlockFull(nn.Module):
    """Full-scale Pairformer block with gradient checkpointing.

    Gradient checkpointing trades memory for compute:
    - Forward pass: don't store activations
    - Backward pass: recompute activations from saved checkpoints
    """
    def __init__(self, c_z=128, c_s=384, n_heads=4, c_hidden=32, use_checkpoint=True):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.c_z = c_z

        # Triangle operations (simplified Linear versions here)
        self.tri_start = nn.Sequential(nn.LayerNorm(c_z), nn.Linear(c_z, c_z))
        self.tri_end   = nn.Sequential(nn.LayerNorm(c_z), nn.Linear(c_z, c_z))
        self.tri_out   = nn.Sequential(nn.LayerNorm(c_z), nn.Linear(c_z, c_z))
        self.tri_in    = nn.Sequential(nn.LayerNorm(c_z), nn.Linear(c_z, c_z))
        self.pair_ff = nn.Sequential(
            nn.LayerNorm(c_z), nn.Linear(c_z, c_z*4), nn.ReLU(), nn.Linear(c_z*4, c_z)
        )
        self.single_ff = nn.Sequential(
            nn.LayerNorm(c_s), nn.Linear(c_s, c_s*2), nn.ReLU(), nn.Linear(c_s*2, c_s)
        )

    def _forward_pair(self, z):
        z = z + self.tri_start(z)
        z = z + self.tri_end(z)
        z = z + self.tri_out(z)
        z = z + self.tri_in(z)
        z = z + self.pair_ff(z)
        return z

    def forward(self, z, s):
        if self.use_checkpoint and z.requires_grad:
            z = checkpoint(self._forward_pair, z, use_reentrant=False)
        else:
            z = self._forward_pair(z)
        s = s + self.single_ff(s)
        return z, s

torch.manual_seed(42)
B, L = 1, 32
c_z, c_s = 64, 128
z = torch.randn(B, L, L, c_z, requires_grad=True)
s = torch.randn(B, L, c_s, requires_grad=True)

block = PairformerBlockFull(c_z=c_z, c_s=c_s, use_checkpoint=True)
z_out, s_out = block(z, s)
loss = z_out.sum() + s_out.sum()
loss.backward()
print(f"Forward: z {z.shape} → {z_out.shape}")
print(f"Gradients: z.grad={z.grad is not None}, s.grad={s.grad is not None}")
print("Gradient checkpointing: saves ~60% memory at ~30% compute overhead")

## 3. Full-Scale Pairformer with Gradient Checkpointing

AF3's Pairformer replaces Evoformer's MSA stack. Each block contains:
1. Triangle attention (starting node) on pair `z`
2. Triangle attention (ending node) on pair `z` — same module, transposed input
3. Triangle multiplicative update (outgoing)
4. Triangle multiplicative update (incoming)
5. Pair transition MLP
6. Single representation update (attends to pair, updates single `s`)

With **gradient checkpointing** (`torch.utils.checkpoint.checkpoint`), we don't store intermediate
activations — they are recomputed during the backward pass. This trades ~40% extra compute for
~N× memory reduction where N = number of blocks.

In [ ]:
import torch
import torch.nn as nn

# Same content as above (duplicate context cell) - show gradient checkpointing benefit
print("Gradient Checkpointing Memory Analysis")
print("=" * 50)

import torch.utils.checkpoint as cp

class DeepPairformer(nn.Module):
    def __init__(self, n_blocks=12, c_z=64, use_ckpt=True):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(nn.LayerNorm(c_z), nn.Linear(c_z, c_z*4),
                          nn.ReLU(), nn.Linear(c_z*4, c_z))
            for _ in range(n_blocks)
        ])
        self.use_ckpt = use_ckpt

    def forward(self, x):
        for block in self.blocks:
            if self.use_ckpt and x.requires_grad:
                x = x + cp.checkpoint(block, x, use_reentrant=False)
            else:
                x = x + block(x)
        return x

torch.manual_seed(42)
B, L, c = 1, 32, 64
x = torch.randn(B, L, L, c, requires_grad=True)

# Without checkpointing
model_no_ckpt = DeepPairformer(n_blocks=8, c_z=c, use_ckpt=False)
out = model_no_ckpt(x)
out.sum().backward()
print(f"Output: {out.shape}")
print("Gradient checkpointing: crucial for training 48-block Pairformer on long sequences")
print("Without it: 48 blocks × L² × c_z activations stay in GPU memory during backward")

In [ ]:
import torch

# AMP (Automatic Mixed Precision) with BFloat16
print("Mixed Precision Training: BF16 vs FP16 vs FP32")
print("=" * 50)

import time

def benchmark_matmul(dtype, size=1024, n_iter=10):
    a = torch.randn(size, size, dtype=dtype, device='cpu')
    b = torch.randn(size, size, dtype=dtype, device='cpu')
    t0 = time.time()
    for _ in range(n_iter):
        c = a @ b
    return (time.time() - t0) / n_iter * 1000  # ms

print(f"{'Dtype':<10} {'Range':<25} {'Precision':<15} {'Speed(ms)'}")
configs = [
    (torch.float32, '~1.18e-38 to 3.4e38', '~7 decimal digits'),
    (torch.float16, '~6e-5 to 65504', '~3 decimal digits'),
    (torch.bfloat16,'~1.18e-38 to 3.4e38','~2 decimal digits'),
]
for dtype, rng, prec in configs:
    ms = benchmark_matmul(dtype)
    print(f"{str(dtype):<10} {rng:<25} {prec:<15} {ms:.2f}")

print()
print("AF3 uses BF16 (not FP16) because:")
print("  BF16 has same exponent range as FP32 → no overflow in attention softmax")
print("  FP16 overflows easily (max=65504) → requires loss scaling hacks")
print("  BF16 precision is lower but sufficient for most ops")
print()
print("GradScaler is NOT needed with BF16 (no overflow risk)")

## 4. Mixed Precision (AMP) Training Setup

AF3 uses **bfloat16** (not float16) on TPUs and modern GPUs. Key differences:
- `bfloat16`: same exponent range as float32 (8 bits), reduced mantissa (7 bits) → numerically stable
- `float16`: smaller exponent range → requires loss scaling to avoid underflow

PyTorch's `torch.cuda.amp.autocast` handles the cast automatically.

In [ ]:
import torch
import torch.nn as nn

# AMP training loop example
torch.manual_seed(42)

class SmallPairformer(nn.Module):
    def __init__(self, c_z=64, n_blocks=4):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(nn.LayerNorm(c_z), nn.Linear(c_z, c_z*2),
                          nn.GELU(), nn.Linear(c_z*2, c_z))
            for _ in range(n_blocks)
        ])
        self.head = nn.Linear(c_z, 1)

    def forward(self, z):
        for block in self.blocks:
            z = z + block(z)
        return self.head(z.mean(dim=[1,2]))

model = SmallPairformer(c_z=32, n_blocks=4)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# AMP training step
L, c = 16, 32
for step in range(3):
    z = torch.randn(2, L, L, c)
    y = torch.randn(2, 1)

    with torch.autocast(device_type='cpu', dtype=torch.bfloat16):
        pred = model(z)
        loss = nn.MSELoss()(pred, y)

    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()
    print(f"Step {step+1}: loss={loss.item():.4f}, pred dtype={pred.dtype}")

print("\nAMP: forward/loss in BF16 (faster), backward in FP32 (stable)")

## 5. Chemical Component Dictionary (CCD)

The CCD is a comprehensive dictionary of ~100K+ small molecule components used in PDB structures.
AF3 uses it to handle ligands that cannot be described by a simple amino acid sequence.

**CCD entry format** (mmCIF subset):
```
data_ATP                         ← 3-char component ID
_chem_comp.id                ATP
_chem_comp.name               "ADENOSINE-5'-TRIPHOSPHATE"
_chem_comp.type               NON-POLYMER
_chem_comp.formula            "C10 H16 N5 O13 P3"
_chem_comp.pdbx_smiles        ...SMILES string...

loop_
_chem_comp_atom.atom_id       ← atom names
_chem_comp_atom.type_symbol   ← element symbols
_chem_comp_atom.pdbx_leaving_atom_flag
...

loop_
_chem_comp_bond.atom_id_1     ← bond table
_chem_comp_bond.atom_id_2
_chem_comp_bond.value_order   ← SING, DOUB, TRIP, AROM
...
```

**Key insight**: CCD gives AF3 the **connectivity graph** and **atom typing** for any ligand.
SMILES alone doesn't give you named atoms (N1, C2, etc.) which are needed to match experimental density.

In [ ]:
# Chemical Component Dictionary (CCD)
print("Chemical Component Dictionary (CCD)")
print("=" * 50)

# CCD entry example (simplified)
ccd_entry = {
    'comp_id': 'ATP',
    'name': 'ADENOSINE-5-TRIPHOSPHATE',
    'formula': 'C10 H16 N5 O13 P3',
    'formula_weight': 507.18,
    'atoms': [
        ('P1',  'P',  'tetrahedron', [-1.2, 0.0, 0.0]),
        ('O1A', 'O',  'trigonal',    [-1.5, 1.3, 0.0]),
        ('O1B', 'O',  'trigonal',    [-1.5,-1.3, 0.0]),
        ('O1', 'O',   'linear',      [ 0.0, 0.0, 0.0]),
        ('C5*','C',   'tetrahedral', [ 1.2, 0.0, 0.5]),
        ('N9', 'N',   'trigonal',    [ 2.5, 0.0, 0.5]),
    ],
    'bonds': [('P1','O1A'), ('P1','O1B'), ('P1','O1'),
              ('O1','C5*'), ('C5*','N9')],
}

print(f"CCD ID: {ccd_entry['comp_id']}")
print(f"Name: {ccd_entry['name']}")
print(f"Formula: {ccd_entry['formula']}")
print(f"MW: {ccd_entry['formula_weight']} Da")
print(f"Heavy atoms: {len(ccd_entry['atoms'])}")
print(f"Bonds: {len(ccd_entry['bonds'])}")
print()
print("In AF3: each CCD heavy atom → 1 token")
print(f"ATP has {len(ccd_entry['atoms'])} heavy atoms → {len(ccd_entry['atoms'])} tokens")
print()
print("CCD provides:")
print("  * Ideal bond lengths and angles for violation loss")
print("  * Atom types for element embedding")
print("  * Chirality information")
print("  * ~100,000+ entries covering all PDB ligands")

## 6. Ligand Atom Graph Construction

AF3 represents ligands as **atom-level tokens**. Unlike proteins (one token per residue),
ligands get one token per **heavy atom**. This allows the diffusion module to predict
individual atom positions rather than residue-level rigid bodies.

**Node features**: element type (one-hot), formal charge, is_aromatic, is_in_ring, degree
**Edge features**: bond order (one-hot), is_aromatic, is_in_ring, distance (from idealized coords)

AF3 uses 128 element types (periodic table up to Og). We use a simplified set here.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def build_ligand_atom_graph(smiles_atoms, smiles_bonds, atom_types=None):
    """Build atom-level graph from CCD/SMILES data.

    AF3 represents ligands as fully-connected atom graphs
    (unlike proteins which use spatial radius cutoff).
    """
    n_atoms = len(smiles_atoms)
    ELEMENTS = ['C','N','O','S','P','F','Cl','Br','I','H','other']
    elem2i = {e:i for i,e in enumerate(ELEMENTS)}

    # Node features: element type one-hot
    x = torch.zeros(n_atoms, len(ELEMENTS))
    for i, atom in enumerate(smiles_atoms):
        idx = elem2i.get(atom, elem2i['other'])
        x[i, idx] = 1.0

    # Edges: bonds + full connectivity (AF3 uses all-pairs for ligands)
    # Bond edges
    edge_index = []
    edge_attr = []
    BOND_TYPES = {'single': 0, 'double': 1, 'triple': 2, 'aromatic': 3}
    for i, j, bond_type in smiles_bonds:
        edge_index += [[i, j], [j, i]]
        bt = BOND_TYPES.get(bond_type, 0)
        edge_index_oh = [0]*4; edge_index_oh[bt] = 1
        edge_attr += [edge_index_oh, edge_index_oh]

    return x, torch.tensor(edge_index).T if edge_index else torch.zeros(2,0,dtype=torch.long),            torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros(0,4)

# Example: ATP simplified
atoms = ['P','O','O','O','O','C','C','N']
bonds = [(0,1,'single'),(0,2,'double'),(0,3,'single'),(3,4,'single'),
         (4,5,'single'),(5,6,'single'),(6,7,'aromatic')]
x, ei, ea = build_ligand_atom_graph(atoms, bonds)
print(f"Ligand graph: {len(atoms)} atoms → {x.shape} node features")
print(f"Edges: {ei.shape[1]} (bonds × 2 for undirected)")
print(f"Edge features: {ea.shape} (bond type one-hot)")
print("AF3 processes ligand atoms at atom-level (1 token/atom) vs protein (1 token/residue)")

## 7. Ligand Tokenization — Matching AF3's Token Sequence

In AF3's unified token sequence:
- Protein residues → 1 token each ("residue-level token")
- DNA/RNA nucleotides → 1 token each
- Ligand heavy atoms → 1 token **each** ("atom-level tokens")

ATP (C10 H16 N5 O13 P3) has 31 heavy atoms → 31 tokens.
A 200-residue protein + ATP complex → 200 + 31 = 231 tokens.

Each ligand token gets an embedding from its element type and chemical environment.

In [ ]:
import torch
import numpy as np
from collections import OrderedDict

def build_af3_token_sequence(protein_chains, ligands):
    """Build AF3's unified token sequence for a complex.

    Token order: protein chains (in order) → ligand atoms (in order)
    Each token has: token_type, residue_idx, chain_idx, atom_name
    """
    tokens = []
    token_id = 0

    for chain_idx, (chain_id, sequence) in enumerate(protein_chains.items()):
        for res_idx, aa in enumerate(sequence):
            tokens.append(OrderedDict([
                ('token_id', token_id),
                ('token_type', 'protein'),
                ('chain', chain_id),
                ('chain_idx', chain_idx),
                ('residue_idx', res_idx),
                ('aa', aa),
                ('atom_name', 'CA'),  # Cα-based token
            ]))
            token_id += 1

    for lig_idx, (lig_id, atoms) in enumerate(ligands.items()):
        for atom_idx, (atom_name, element) in enumerate(atoms):
            tokens.append(OrderedDict([
                ('token_id', token_id),
                ('token_type', 'ligand'),
                ('chain', lig_id),
                ('chain_idx', len(protein_chains) + lig_idx),
                ('residue_idx', atom_idx),
                ('aa', element),
                ('atom_name', atom_name),
            ]))
            token_id += 1

    return tokens

chains = {'A': 'ACDEFGHIKLM', 'B': 'MNPQRSTVWY'}
ligands = {'LIG': [('P1','P'),('O1','O'),('O2','O'),('C1','C'),('N1','N')]}

tokens = build_af3_token_sequence(chains, ligands)
print(f"Total tokens: {len(tokens)}")
from collections import Counter
type_counts = Counter(t['token_type'] for t in tokens)
print(f"  Protein tokens: {type_counts['protein']} (1 per residue)")
print(f"  Ligand tokens:  {type_counts['ligand']} (1 per heavy atom)")
print()
for t in tokens[-3:]:
    print(f"  Token {t['token_id']}: {t['token_type']} chain={t['chain']} atom={t['atom_name']}")

## 8. lDDT-PLI — Ligand Confidence Score

**lDDT** measures local distance difference for Cα atoms.
**lDDT-PLI** (Protein–Ligand Interaction) measures local distance difference specifically
for contacts between **protein atoms** and **ligand atoms** within a cutoff distance.

AF3's confidence head predicts lDDT-PLI per ligand atom.
A good pose has lDDT-PLI > 0.6 (at least 60% of protein-ligand contacts preserved).

Formula (same as lDDT but restricted to cross-molecule pairs):
$$\text{lDDT-PLI}_i = \frac{1}{|\mathcal{N}_i|} \sum_{j \in \mathcal{N}_i} \frac{1}{4}\sum_{t \in \{0.5,1,2,4\}} \mathbf{1}[|d^{pred}_{ij} - d^{true}_{ij}| < t]$$

where $\mathcal{N}_i$ = all protein atoms within 6Å of ligand atom $i$ in the true structure.

In [ ]:
import torch
import numpy as np

def lddt_pli(pred_ligand_coords, true_ligand_coords,
             pred_protein_coords, true_protein_coords,
             interface_cutoff=6.0,
             thresholds=(0.5, 1.0, 2.0, 4.0)):
    """lDDT-PLI: lDDT for Protein-Ligand Interactions.

    Evaluates the accuracy of predicted ligand-protein contacts.
    Counts only (ligand_atom, protein_atom) pairs where true distance < cutoff.
    """
    scores = []
    n_lig = len(pred_ligand_coords)
    n_prot = len(pred_protein_coords)

    true_lig  = torch.tensor(true_ligand_coords, dtype=torch.float)
    pred_lig  = torch.tensor(pred_ligand_coords, dtype=torch.float)
    true_prot = torch.tensor(true_protein_coords, dtype=torch.float)
    pred_prot = torch.tensor(pred_protein_coords, dtype=torch.float)

    for li in range(n_lig):
        for pi in range(n_prot):
            true_d = (true_lig[li] - true_prot[pi]).norm().item()
            if true_d > interface_cutoff:
                continue
            pred_d = (pred_lig[li] - pred_prot[pi]).norm().item()
            diff = abs(pred_d - true_d)
            score = np.mean([diff < t for t in thresholds])
            scores.append(score)

    lddt_pli_score = np.mean(scores) if scores else 0.0
    return lddt_pli_score, len(scores)

np.random.seed(42)
# Simulate a ligand (10 atoms) near protein (30 residues)
true_lig  = np.random.randn(10, 3)
true_prot = np.random.randn(30, 3)
pred_lig_good  = true_lig  + np.random.randn(10, 3) * 0.3
pred_lig_bad   = true_lig  + np.random.randn(10, 3) * 3.0

score_good, n_pairs = lddt_pli(pred_lig_good, true_lig, true_prot, true_prot)
score_bad,  _       = lddt_pli(pred_lig_bad,  true_lig, true_prot, true_prot)
print(f"Interface pairs evaluated: {n_pairs}")
print(f"lDDT-PLI (good prediction): {score_good:.3f}")
print(f"lDDT-PLI (bad prediction):  {score_bad:.3f}")
print("lDDT-PLI > 0.7 = good ligand binding pose prediction")

## 9. Integration: Full Inference Pipeline Sketch

Putting it all together: how the components connect in an AF3-style inference run.

In [ ]:
import torch
import torch.nn as nn

# AF3 Full Inference Pipeline Sketch
print("AF3-Style Inference Pipeline")
print("=" * 60)
print()
print("Input:")
print("  - Protein sequence(s) + DNA/RNA + ligand SMILES")
print()
print("Step 1: Feature extraction")
print("  a. MSA search (Jackhmmer vs UniRef90, MGnify)")
print("  b. Template search (HHsearch vs PDB)")
print("  c. CCD lookup for ligand atoms")
print("  d. Build Atom14/Atom37 features")
print()
print("Step 2: Input embedder")
print("  - Residue/atom type embeddings → single repr s (L, c_s)")
print("  - Pair features from distances/orientations → pair repr z (L, L, c_z)")
print("  - MSA → outer product mean → z update")
print()
print("Step 3: Pairformer (48 blocks)")
print("  - Triangle attention (starting + ending)")
print("  - Triangle multiplicative update (outgoing + incoming)")
print("  - Pair transition (2-layer MLP)")
print("  - Single repr update")
print()
print("Step 4: Diffusion module")
print("  - Sample x_T ~ N(0, I) (all-atom coordinates)")
print("  - Iteratively denoise: x_{t-1} = denoise(x_t, z, s, t)")
print("  - 200 diffusion steps (inference: 1000 steps for quality)")
print()
print("Step 5: Confidence prediction")
print("  - pLDDT per atom (0-100)")
print("  - PAE matrix (L, L)")
print("  - pTM / ipTM scores")
print()
print("Output: all-atom 3D coordinates + confidence scores")

# Minimal pipeline class sketch
class AF3Pipeline(nn.Module):
    def __init__(self, c_z=128, c_s=384, n_pairformer=48):
        super().__init__()
        self.input_embedder = nn.Linear(21, c_s)   # AA + gap → single
        self.pair_init = nn.Linear(c_s*2, c_z)
        print(f"\nPipeline initialized: c_z={c_z}, c_s={c_s}, n_blocks={n_pairformer}")
        print(f"Parameters (embedder only): {sum(p.numel() for p in self.parameters()):,}")

pipeline = AF3Pipeline(c_z=64, c_s=128, n_pairformer=4)

## 10. Q&A

**Q1: Why does AF3 use 48 Pairformer blocks when AlphaFold2 used 48 Evoformer blocks?**

The same depth. Pairformer is the AF3 equivalent of Evoformer's pair stack — it drops the MSA row/column attention (MSA is handled separately in the input embedding) and instead runs the 4 triangle operations + pair transition + single update. Empirically 48 blocks was the sweet spot between capacity and compute.

**Q2: What is the memory complexity of a single triangle attention pass?**

Without chunking: O(N³) for the attention weights (each of N² pairs attends to N positions). With chunking at chunk size c: O(N² × c) at any one time. For N=384, c=64: 384² × 64 × 2 bytes ≈ 19 MB vs 384³ × 2 bytes ≈ 113 MB — 6× reduction.

**Q3: What does the CCD provide beyond a SMILES string?**

Three concrete things: (1) **Named atoms** — CCD gives atom IDs (N1, C2, PA, O1A) which map to experimental electron density. SMILES gives anonymous atoms. (2) **Leaving atoms** — the `pdbx_leaving_atom_flag` marks atoms that detach during polymerization (e.g., phosphates in nucleotides). (3) **Idealized 3D coordinates** — SMILES encodes topology only; CCD provides an idealized geometry used to initialize diffusion.

**Q4: Why does AF3 use bfloat16 instead of float16 for mixed precision?**

The exponent range. `bfloat16` has 8 exponent bits (same as float32, range ±3.4×10³⁸); `float16` has only 5 exponent bits (range ±6.5×10⁴). Triangle multiplicative updates sum over N products — these sums can exceed float16's range, causing NaN/inf without loss scaling. bfloat16 handles the same range as float32 with no loss scaling needed.

**Q5: How does gradient checkpointing interact with the recycling loop?**

Recycling uses `torch.no_grad()` for all passes except the last. Gradient checkpointing only matters inside the final pass where gradients flow. So the pattern is: `no_grad()` for passes 1 to N-1 (no memory for activations at all), then the final pass with `checkpoint` enabled on each Pairformer block. Combined, you get near-constant memory regardless of N_recycles or N_blocks.

**Q6: What AF3 components are still NOT covered by this notebook series?**

- **Diffusion Transformer**: the full denoising U-Net that generates 3D coordinates from z and s
- **Template featurization**: aligning query to PDB templates, extracting torsion/pair features
- **MSA pairing across species**: handling multi-chain MSAs correctly (paired + unpaired MSAs)
- **Atom cross-attention** (`atom_cross_attention.py`): maps between atom-level and residue-level representations
- **Full violation loss** with all AF3 bond geometry tables (CSD-based bonds, not just backbone)

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Gradient checkpointing**: recompute activations during backward pass instead of storing — O(√N) memory vs O(N), O(N) extra compute
- **Mixed precision IEEE 754**: bfloat16 has same 8-bit exponent as float32 (no overflow risk), 7-bit mantissa; float16 has 5-bit exponent (overflows at 65504)
- **ZeRO optimizer**: Stage 1 shards optimizer states, Stage 2 + gradients, Stage 3 + parameters — linear memory reduction with world size
- **Tensor parallelism**: split weight matrices column/row-wise across GPUs — Megatron-LM style
- **CCD molecular graph**: Chemical Components Dictionary — SMILES + bond topology for all PDB ligands, used for Atom14 ligand extension

### 2️⃣ Must-Have Popular Resources
- 📘 ZeRO (Rajbhandari 2020, arXiv 1910.02054) — the foundational distributed training paper; read Sections 1–3
- 📘 FlashAttention-2 (Dao 2023) — IO-aware exact attention; mandatory reading for anyone training transformers
- 📘 Chinchilla (Hoffmann 2022) — compute-optimal scaling laws; defines how to allocate GPU budget
- 🎓 Designing ML Systems (Chip Huyen) — production ML infrastructure, training pipelines
- ⭐ GitHub [microsoft/DeepSpeed](https://github.com/microsoft/DeepSpeed) 35k★ — ZeRO, pipeline parallelism, sparse attention
- ⭐ GitHub [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) 14k★ — drop-in fast attention

### 3️⃣ Practicals — Hands-On
- 💻 Profile Pairformer memory with torch.profiler — compare baseline vs gradient checkpointing at N=128, 256, 512
- 💻 Implement chunked triangle attention — process pair representations in chunks to reduce peak memory
- 💻 Set up DDP training on 2 GPUs with torch.distributed — verify gradient sync with allreduce hook
- 🤗 HuggingFace: [Accelerate tutorials](https://huggingface.co/docs/accelerate) — distributed training in 3 lines
- 🤗 HuggingFace: [chembl/chembl_34](https://huggingface.co/datasets/chembl/chembl_34) — 2.4M compounds with SMILES

### 4️⃣ Real-World Problems
- 🏭 AF3 trained on hundreds of Google TPUs — gradient checkpointing + bfloat16 were essential to fit the model
- 🏭 Boltz-2 trained with DeepSpeed ZeRO-3 + gradient checkpointing on A100 cluster
- 📊 Dataset: [ChEMBL 34 on HuggingFace](https://huggingface.co/datasets/chembl/chembl_34) — for CCD ligand featurization
- 🔬 Application: Cost estimation — Chinchilla scaling laws predict optimal model size given TPU-hour budget

### 5️⃣ Interview Tips
- ❓ Must know: Explain gradient checkpointing √N memory trick — recompute N checkpoints, each stores √N activations
- ❓ Must know: Why bfloat16 over float16 for training? (same exponent as float32 → no gradient overflow, critical for large LR)
- ❓ Must know: DDP vs DataParallel — always use DDP (DataParallel has Python GIL bottleneck, single process)
- ⚠️ Common mistake: Using ZeRO Stage 3 for small models — communication overhead exceeds memory savings below ~10B parameters
- 💡 Pro tip: For transformer fine-tuning, gradient checkpointing + bfloat16 + ZeRO-2 is the standard production setup

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) — FlashAttention-3 for Hopper GPUs (H100)
- 🔥 Trending: [linkedin/Liger-Kernel](https://github.com/linkedin/Liger-Kernel) — fused triton kernels for transformer training (2x throughput)
- 🔥 Trending: [google/jax](https://github.com/google/jax) 30k★ — AF3 and Boltz use JAX for TPU training
- 🚀 Build this: A memory-efficient Pairformer training loop using gradient checkpointing + AMP that processes N=512 residues on a single 24GB GPU

In [ ]:
# Module 07/04 Resources
print("Module 07/04 — AF3 Full Scale — Key References")
print("=" * 60)
print()
refs = {
    "Gradient Checkpointing": [
        "Chen et al. 2016 — Training Deep Nets with Sublinear Memory Cost",
        "torch.utils.checkpoint docs",
    ],
    "Mixed Precision": [
        "Micikevicius et al. 2018 — Mixed Precision Training",
        "torch.autocast docs (BF16 preferred for TPU/Ampere+)",
    ],
    "CCD": [
        "wwPDB Chemical Component Dictionary: wwpdb.org/data/ccd",
        "~100k entries, updated monthly",
        "AF3 uses CCD for ligand bond geometry in violation loss",
    ],
    "AF3 System": [
        "Abramson et al. 2024 Nature — AlphaFold3",
        "github.com/google-deepmind/alphafold3 — weights + inference code",
        "AF3 server: alphafoldserver.com",
    ],
    "Related Models": [
        "Boltz-1/2: open-source AF3 alternative (MIT license)",
        "Chai-1: commercial AF3 alternative",
        "RoseTTAFold All-Atom: UW/IPD",
    ],
}
for cat, items in refs.items():
    print(f"{cat}:")
    for item in items:
        print(f"  * {item}")
    print()

## Real World Problems

### Problem 1: Reproduce AF3 Pairformer Memory Benchmarks

The AF3 paper reports training on crops of N=384 residues/atoms. Using the `estimate_pairformer_memory` function and the chunked Pairformer implementation:
1. Determine the minimum GPU VRAM (GB) needed for N=384, batch=2, bf16, WITH gradient checkpointing and chunked attention (chunk=64).
2. Profile the actual memory on a GPU if available using `torch.cuda.max_memory_allocated()`.
3. Why does AF3 choose N_crop=384 rather than 512? Estimate the memory difference.

**Resources**:
- [AF3 source (JAX/Haiku)](https://github.com/google-deepmind/alphafold3)
- [OpenFold PyTorch reimplementation](https://github.com/aqlaboratory/openfold)
- [PyTorch AMP documentation](https://pytorch.org/docs/stable/amp.html)

---

### Problem 2: Parse and Visualize a Real CCD Entry

Download the full CCD components file from the PDB and parse a ligand of your choice (HEM, GTP, or FAD):
1. Count the number of heavy atoms and bonds.
2. Build the atom graph using `ccd_to_atom_graph`.
3. Identify the aromatic rings using the `pdbx_aromatic_flag` field in the bond table.
4. What is the degree distribution of carbon atoms in the ligand?

**Resources**:
- [CCD full components download](https://www.wwpdb.org/data/ccd)
- [AF3 chemical_components.py](https://github.com/google-deepmind/alphafold3/blob/main/src/alphafold3/structure/chemical_components.py)
- [RDKit for SMILES visualization](https://github.com/rdkit/rdkit)

---

### Problem 3: lDDT-PLI on Real AF3 Predictions

The PLINDER benchmark provides protein-ligand complexes with AF3 predictions.
1. Download 3 AF3-predicted structures from PLINDER.
2. Extract protein Cβ positions and ligand heavy atom positions from the mmCIF files.
3. Compute lDDT-PLI using the implementation above.
4. Compare your scores to the values reported in the PLINDER paper.

**Resources**:
- [PLINDER benchmark (HuggingFace)](https://huggingface.co/datasets/plinder-org/plinder)
- [PoseBusters validation toolkit (GitHub)](https://github.com/maabuu/posebusters)
- [AF3 confidence score paper](https://www.nature.com/articles/s41586-024-07487-w)

---

### Problem 4: Kaggle — Leash Bio Predict New Medicines by Mapping DNA

This competition requires predicting binding of 98M+ small molecules to protein targets. The molecules are given as SMILES strings. Apply what you've learned about ligand atom graphs:
1. Convert SMILES to atom-level node/edge features using `ccd_to_atom_graph` logic (or RDKit).
2. Build a message-passing GNN (similar to `LigandAtomEmbedder`) to predict binding.
3. Ablate: how much does adding bond order one-hot vs. just SING/DOUB/AROM matter?

**Resources**:
- [Leash Bio competition (Kaggle)](https://www.kaggle.com/competitions/leash-BELKA)
- [Molecular property prediction (HuggingFace)](https://huggingface.co/datasets/chembl/chembl_34)
- [PyTorch Geometric for GNNs (GitHub)](https://github.com/pyg-team/pytorch_geometric)

---

| Resource | Type | Use |
|---|---|---|
| [google-deepmind/alphafold3](https://github.com/google-deepmind/alphafold3) | GitHub | Official AF3 source (JAX) |
| [aqlaboratory/openfold](https://github.com/aqlaboratory/openfold) | GitHub | PyTorch AF2/AF3 reference |
| [pyg-team/pytorch_geometric](https://github.com/pyg-team/pytorch_geometric) | GitHub | GNN framework for ligand graphs |
| [plinder-org/plinder](https://huggingface.co/datasets/plinder-org/plinder) | HuggingFace | Protein-ligand benchmark |
| [chembl/chembl_34](https://huggingface.co/datasets/chembl/chembl_34) | HuggingFace | 2.4M bioactive molecules |
| [Leash Bio BELKA](https://www.kaggle.com/competitions/leash-BELKA) | Kaggle | 98M molecule binding prediction |
| [NovaSky AF3 Kaggle](https://www.kaggle.com/competitions/stanford-ribonanza-rna-folding) | Kaggle | RNA structure prediction (AF3 approach) |

## Mastery Check

On a first pass, success means you can:
1. explain what this notebook's component does in the larger AF3 pipeline
2. describe why geometry matters here
3. explain at least one confidence or training metric in words
4. decide whether you should revisit this notebook later for deeper implementation work


---
## 📚 Resources — Full-Scale AF3 & Ligand Handling

### University + Industry Resources (All Free)
| Resource | What You Get |
|----------|--------------|
| **[MIT 6.874 Computational Biology of Disease](https://mit6874.github.io/)** | How AF3 is used in drug discovery; guest lectures from DeepMind/Isomorphic |
| **[Flash Attention paper](https://arxiv.org/abs/2205.14135)** | The memory-efficient attention that makes large N tractable |
| **[Flash Attention 2](https://arxiv.org/abs/2307.08691)** | Further optimizations; what Boltz-2 uses |
| **[PLINDER benchmark paper](https://www.biorxiv.org/content/10.1101/2024.07.17.603955v3)** | 449K protein-ligand structures; real benchmark for your models |

### Chemical Component Dictionary (CCD)
- **[wwPDB CCD Browser](https://www.wwpdb.org/ccd.html)** — browse all ~100K ligand definitions
- **[RDKit Documentation](https://www.rdkit.org/docs/)** — the Python library for reading CCD/SDF files
- **[OpenBabel](https://openbabel.org/)** — convert between MOL2, SDF, SMILES — essential for ligand processing

### Memory Optimization References (Must-Know)
- **[Gradient Checkpointing Paper](https://arxiv.org/abs/1604.06174)** (Chen et al. 2016) — original activation recomputation paper
- **[Mixed Precision Training](https://arxiv.org/abs/1710.03740)** (Micikevicius et al. 2018) — BF16 vs FP16 tradeoffs
- **[DeepSpeed ZeRO paper](https://arxiv.org/abs/1910.02054)** — model parallelism for models too large for one GPU
- **[Andrej Karpathy nanoGPT](https://github.com/karpathy/nanoGPT)** — best teaching example of production-quality training with all optimizations

### Chunked Attention Explained
```
Full triangle attention:  O(N³) memory — for N=2048, that's 8 billion intermediate values
Chunked triangle attention: O(N²·C) — process C=128 rows at a time → 16x memory reduction
Flash attention (for standard SA): O(N) — uses online softmax trick → even better
```

### Real-World Scale Context
| AF3 Configuration | Memory | Where |
|-------------------|--------|-------|
| Tutorial (N=50) | ~1 GB | Your laptop |
| Small protein (N=256) | ~4 GB | RTX 3080 |
| Typical complex (N=1024) | ~40 GB | A100 80GB |
| Maximum (N=5120) | ~300 GB | TPU pod / multi-GPU |


## 🐛 Debug Exercise — Triangle Attention & Pair Update

Find and fix the **3 bugs** in the Pairformer-style update below.

**Expected correct output:**
```
Masked attention weights: zeros where mask is False (before softmax)
Pair bias added BEFORE softmax for correct numerical behavior
Outer product mean output shape: [L, L, c_z]
```

<details>
<summary>Show answer</summary>

**Bug 1 — Mask applied after softmax:** The attention mask (e.g., padding mask) must be
applied *before* softmax by adding a large negative value to masked positions.
Applying it after softmax sets some weights to zero but the remaining weights no longer
sum to 1. Fix: `logits = logits + mask * -1e9` then `softmax(logits)`.

**Bug 2 — Pair bias added after attention:** The pair representation bias (b_ij) should be
added to the attention logits *before* softmax, not to the output after.
Fix: `logits = q @ k.T / sqrt(d) + pair_bias` then `softmax(logits)`.

**Bug 3 — Wrong outer product mean dimension:** The outer product of MSA row vectors
should produce shape `[L, L, c_m * c_m]` (or projected to c_z), not `[L, c_m, c_m]`.
Fix: `outer = a.unsqueeze(1) * b.unsqueeze(0)` then flatten/project.
</details>


In [ ]:
# DEBUG EXERCISE — Triangle attention and pair update, find and fix 3 bugs
import torch
import torch.nn.functional as F
import math

torch.manual_seed(42)
L, heads, d_head, c_z = 8, 4, 16, 32

# --- Bug 1: mask applied AFTER softmax ---
q = torch.randn(heads, L, d_head)
k = torch.randn(heads, L, d_head)
v = torch.randn(heads, L, d_head)
pair_bias = torch.randn(heads, L, L)   # pair representation bias b_ij

# Valid positions (True = attend, False = mask out)
mask = torch.ones(L, L, dtype=torch.bool)
mask[3:, 3:] = False   # mask out bottom-right quadrant

logits = torch.einsum('hid,hjd->hij', q, k) / math.sqrt(d_head)

# Bug 2: pair bias added AFTER softmax instead of before
attn_weights = F.softmax(logits, dim=-1)         # softmax WITHOUT pair bias — wrong
attn_weights = attn_weights + pair_bias          # BUG 2: bias added post-softmax
                                                  # Fix: logits = logits + pair_bias; then softmax

# Bug 1: mask applied after softmax — weights no longer sum to 1
attn_weights = attn_weights * mask.unsqueeze(0)  # BUG 1: multiply by 0/1 post-softmax
print("Attention weights sum (row 0, should be 1.0):", attn_weights[0, 0].sum().item())
print("After masking post-softmax, sum is:", attn_weights[0, 0].sum().item(), "— not 1.0!")

out = torch.einsum('hij,hjd->hid', attn_weights, v)
print("Attention output shape:", out.shape)

# --- Bug 3: outer product mean wrong shape ---
c_m = 8   # MSA channel dim
a = torch.randn(L, c_m)   # features for one MSA row position i
b = torch.randn(L, c_m)   # features for one MSA row position j

# BUG 3: this computes [L, c_m, c_m] but we need [L, L, c_m*c_m]
outer_wrong = a.unsqueeze(-1) * b.unsqueeze(-2)    # shape [L, c_m, c_m] — wrong
print(f"\nOuter product shape (buggy): {outer_wrong.shape}")
print(f"Expected outer product shape: [{L}, {L}, {c_m*c_m}]  (for pair update)")

# Fix would be:
# outer_correct = a.unsqueeze(1) * b.unsqueeze(0)   # [L, L, c_m] then project to c_z
